<a href="https://colab.research.google.com/github/Ikbaaal08/Image-Smoothing/blob/main/8_1_transfer_learning_faster_rcnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 8.1 Transfer Learning Faster R-CNN

⚠️⚠️⚠️ *Please open this notebook in Google Colab* by click below link ⚠️⚠️⚠️<br><br>
<a href="https://colab.research.google.com/github/Muhammad-Yunus/Belajar-OpenCV-ObjectDetection/blob/main/Pertemuan%208/8.1%20transfer_learning_faster_rcnn.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><br><br><br>
- Click `Connect` button in top right Google Colab notebook,<br>
<img src="https://github.com/Muhammad-Yunus/Belajar-OpenCV-ObjectDetection/blob/main/Pertemuan%207/resource/cl-connect-gpu.png?raw=1" width="250px">
- If connecting process completed, it will turn to something look like this<br>
<img src="https://github.com/Muhammad-Yunus/Belajar-OpenCV-ObjectDetection/blob/main/Pertemuan%207/resource/cl-connect-gpu-success.png?raw=1" width="250px">

- Check GPU connected into Colab environment is active

In [1]:
!nvidia-smi

Mon Apr 27 03:59:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

- Install necasarry package

In [3]:
!pip install torchmetrics
!pip install onnx
!pip uninstall onnxscript

#### 8.1.1 Download Dataset From Roboflow
- Folow instruction in [4.1 dataset_annotation_roboflow.ipynb](https://github.com/Muhammad-Yunus/Belajar-OpenCV-ObjectDetection/blob/main/Pertemuan%204/4.1%20dataset_annotation_roboflow.ipynb) to prepare `Scissors Dataset` example,
- Open `Roboflow` > `Project` > `Versions` menu
- Then click `Download Dataset`<br>
<img src="https://github.com/Muhammad-Yunus/Belajar-OpenCV-ObjectDetection/blob/main/Pertemuan%208/resource/rb-download-dataset.png?raw=1" width="850px">
- Choose `COCO` format and select `Show download code` then click `Continue` <br>
<img src="https://github.com/Muhammad-Yunus/Belajar-OpenCV-ObjectDetection/blob/main/Pertemuan%208/resource/rb-download-format.png?raw=1" width="350px">
- click `Copy` icon to copy roboflow download code<br>
<img src="https://github.com/Muhammad-Yunus/Belajar-OpenCV-ObjectDetection/blob/main/Pertemuan%208/resource/rb-copy-download-code.png?raw=1" width="350px">
- Then <font color="orange">replace below code</font> using the copied roboflow download code above,


In [8]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="WrS5vNBIFmIJcDFAMiSr")
project = rf.workspace("ikbals-workspace").project("scissors-qy2yh")
version = project.version(2)
dataset = version.download("coco")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to scissors-2 in coco:: 100%|██████████| 25/25 [00:00<00:00, 4496.66it/s]


In [9]:
import torch
import torchvision
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision

import cv2
import json
import os

- Create Pytorch Custom Dataset Loader to load `Scissors Dataset` from Roboflow

In [10]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, root):
        self.root = root

        # Load COCO JSON annotations
        with open(os.path.join(root, "_annotations.coco.json")) as f:
            data = json.load(f)

        # Process images and annotations
        self.images = {img['id']: img for img in data['images'] if img['file_name'].endswith('.jpg')}
        self.annotations = data['annotations']
        self.categories = {cat['id']: cat['name'] for cat in data['categories']}

        # Group annotations by image ID for easy lookup
        self.img_to_anns = {img_id: [] for img_id in self.images.keys()}
        for ann in self.annotations:
            self.img_to_anns[ann['image_id']].append(ann)

        # Store only images that have annotations
        self.imgs = [img_id for img_id in self.img_to_anns if self.img_to_anns[img_id]]

    def __getitem__(self, idx):
        # Get image and annotations for this index
        img_id = self.imgs[idx]
        img_info = self.images[img_id]
        img_path = os.path.join(self.root, img_info['file_name'])
        image = cv2.imread(img_path)  # Load image using OpenCV
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
        image = torch.from_numpy(image).to(torch.float32)  # Convert NumPy array to PyTorch tensor
        image = image.permute(2, 0, 1)  # Change the order of dimensions from (H, W, C) to (C, H, W)
        image = image / 255.0  # Normalize pixel values to [0, 1]

        # Get bounding boxes and labels
        boxes = []
        labels = []
        for ann in self.img_to_anns[img_id]:
            bbox = ann['bbox']
            boxes.append([bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]])  # Convert to [x_min, y_min, x_max, y_max]
            labels.append(ann['category_id'])

        # Convert boxes and labels to tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}

        return image, target

    def __len__(self):
        return len(self.imgs)


- Load Train & Validation Dataset

In [11]:
# Assume dataset and DataLoader for training and validation have been set up
DATASET_ROOT_PATH = dataset.location

dataset_train = CustomDataset(root=f"{dataset.location}/train")
dataset_val = CustomDataset(root=f"{dataset.location}/valid")

data_loader_train = DataLoader(dataset_train, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
data_loader_val = DataLoader(dataset_val, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

print(f"Train Dataset : {len(dataset_train)} data, \t Validation Dataset : {len(dataset_val)} data")


Train Dataset : 15 data, 	 Validation Dataset : 3 data


- Set Dataset Class Labels and Count

In [12]:
def get_dataset_count(root) :
  with open(os.path.join(root, "_annotations.coco.json")) as f:
      data = json.load(f)

  # Extract all category names into a list
  labels = [category["name"] for category in data.get("categories", [])]
  return len(labels), labels


# Define Number of dataset class and class labels based on downloaded Roboflow Dataset
NUM_CLASS, CLASS_LABELS = get_dataset_count(root=f"{dataset.location}/train")

print(f"Number of Class : {NUM_CLASS}, \t Labels : {CLASS_LABELS}")

Number of Class : 2, 	 Labels : ['My-First-Project', 'Gunting']


- Load Pretrained <font color="orange">Faster R-CNN</font> Model with <font color="orange">ResNet-50</font> Backbone from [Torchvision](https://pytorch.org/vision/main/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn_v2.html)
  - Modify number of detection class on pretrained <font color="orange">Faster R-CNN Predictor</font> (a.k.a `FastRCNNPredictor()`)

In [13]:
# Load the pre-trained model
model = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')

# Get the number of input features for the classifier
in_features = model.roi_heads.box_predictor.cls_score.in_features

# Replace the pre-trained head with a new one
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASS)

# Move model to device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


100%|██████████| 167M/167M [00:01<00:00, 158MB/s]


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       

- Run <font color="orange">Training Loop</font> for Pretrained Faster R-CNN with Custom Dataset downloaded from Roboflow
  - Calculate <font color="orange">Loss</font> and <font color="orange">mAP</font> on each Epoch
  - Run training loop at least for 20 epoch

In [14]:
# Initialize mAP metric
map_metric = MeanAveragePrecision(iou_thresholds=[0.5, 0.75, 0.9],  # IoU thresholds
                                  class_metrics=True)  # Separate mAP per class


# Define SGD Optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)


# Define Evaluation function to compute mAP on Evaulation dataset
def evaluate(model, validation_loader, device):
    model.eval()  # Ensure model is in eval mode

    for images, targets in validation_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        with torch.no_grad():
            outputs = model(images)

        # Format predictions and targets for torchmetrics
        preds = [
            {
                "boxes": output["boxes"].cpu(),
                "scores": output["scores"].cpu(),
                "labels": output["labels"].cpu(),
            }
            for output in outputs
        ]

        gts = [
            {
                "boxes": target["boxes"].cpu(),
                "labels": target["labels"].cpu(),
            }
            for target in targets
        ]

        # Update mAP metric with predictions and ground truths
        map_metric.update(preds, gts)

    # Compute mAP at the end of the epoch
    map_result = map_metric.compute()
    return map_result




# Training loop
NUM_EPOCH = 20
for epoch in range(NUM_EPOCH):
    model.train()

    total_loss = 0
    for images, targets in data_loader_train:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_loss += losses.item()

        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    # Validation step
    mAP = evaluate(model, data_loader_val, device)
    print(f"Epoch {epoch+1}, \tLoss: {total_loss/len(data_loader_train)}, \tmAP: {mAP['map']:.4f}, \tmAP-50: {mAP['map_50']:.4f}")


Epoch 1, 	Loss: 0.5211260169744492, 	mAP: 0.2154, 	mAP-50: 0.6463
Epoch 2, 	Loss: 0.2993590086698532, 	mAP: 0.1304, 	mAP-50: 0.3911
Epoch 3, 	Loss: 0.27526795119047165, 	mAP: 0.1723, 	mAP-50: 0.4574
Epoch 4, 	Loss: 0.22510217130184174, 	mAP: 0.1585, 	mAP-50: 0.4177
Epoch 5, 	Loss: 0.1841539740562439, 	mAP: 0.1377, 	mAP-50: 0.4009
Epoch 6, 	Loss: 0.16081949695944786, 	mAP: 0.1594, 	mAP-50: 0.4103
Epoch 7, 	Loss: 0.13457446917891502, 	mAP: 0.1970, 	mAP-50: 0.4462
Epoch 8, 	Loss: 0.12233378551900387, 	mAP: 0.2358, 	mAP-50: 0.5161
Epoch 9, 	Loss: 0.10302349552512169, 	mAP: 0.2866, 	mAP-50: 0.5763
Epoch 10, 	Loss: 0.09757434949278831, 	mAP: 0.3117, 	mAP-50: 0.6241
Epoch 11, 	Loss: 0.07872080057859421, 	mAP: 0.3372, 	mAP-50: 0.6567
Epoch 12, 	Loss: 0.069918317720294, 	mAP: 0.3550, 	mAP-50: 0.6815
Epoch 13, 	Loss: 0.062309516593813896, 	mAP: 0.3736, 	mAP-50: 0.7036
Epoch 14, 	Loss: 0.05738705396652222, 	mAP: 0.3940, 	mAP-50: 0.7263
Epoch 15, 	Loss: 0.0657557612285018, 	mAP: 0.3899, 	mAP-50: 0

- Export trained faster R-CNN model into <font color="orange">ONNX Format</font>

In [15]:
# Load model
model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(pretrained=True)
model.eval()

MODEL_NAME = "scissors.onnx"

# Dummy input (TANPA batch dimension!)
dummy = torch.randn(3, 800, 800)

# Export
torch.onnx.export(
    model,
    ([dummy],),   # ⬅️ TRIK: tetap list tapi tanpa wrapper
    MODEL_NAME,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["images"],
    output_names=["output"],  # ⬅️ harus 1 output (karena dict)
    dynamo=False              # ⬅️ WAJIB
)

The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT` to get the most up-to-date weights.
You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
To copy construc

- Download Trained Faster R-CNN ONNX Model into disk.

In [16]:
from google.colab import files

files.download(MODEL_NAME)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>